# **MNIST centralized**

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn.functional as F 
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from torch.utils.data import DataLoader

## Define any transforms you need; at minimum, convert to Tensor

In [ ]:
transform = transforms.ToTensor()

# This will download MNIST into ./data if it’s not already there

In [ ]:
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

In [ ]:
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [ ]:
print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

## Load MNIST and preprocess

In [ ]:
transform = transforms.ToTensor()
train_ds_full = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
X_full = train_ds_full.data.numpy().reshape(-1, 28*28).astype(np.float32) / 255.0
y_full = train_ds_full.targets.numpy().astype(np.int64)
X_test = test_ds.data.numpy().reshape(-1, 28*28).astype(np.float32) / 255.0
y_test = test_ds.targets.numpy().astype(np.int64)

## Train/val split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=42
)

## To Tensors & DataLoaders

In [ ]:
def to_loader(X, y, batch_size=64, shuffle=False):
    tX = torch.from_numpy(X)
    ty = torch.from_numpy(y)
    ds = TensorDataset(tX, ty)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader   = to_loader(X_val,   y_val)
test_loader  = to_loader(X_test,  y_test)

## Model (same as yours, final layer → 10)

In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self, mlp_hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, mlp_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(mlp_hidden, mlp_hidden//2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(mlp_hidden//2, 10),
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

## Loss & optimizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MNISTMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

## Training / evaluation loops

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss, correct = 0.0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(Xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == yb).sum().item()
    return total_loss/len(loader.dataset), correct/len(loader.dataset)

In [ ]:
def eval_epoch(loader):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out = model(Xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * Xb.size(0)
            correct += (out.argmax(dim=1) == yb).sum().item()
    return total_loss/len(loader.dataset), correct/len(loader.dataset)

history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

best_val, wait, patience = float('inf'), 0, 5
for epoch in range(1, 21):
    tr_loss, tr_acc = train_epoch(train_loader)
    va_loss, va_acc = eval_epoch(val_loader)
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    print(f"Epoch {epoch} | Train loss {tr_loss:.4f}, acc {tr_acc:.3f} | Val loss {va_loss:.4f}, acc {va_acc:.3f}")
    if va_loss < best_val:
        best_val, wait = va_loss, 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

## Plot loss & accuracy

In [ ]:
epochs = range(1, len(history["train_loss"])+1)
plt.figure(figsize=(10, 5))
plt.figure(); plt.plot(epochs, history["train_loss"], label='Train loss'); plt.plot(epochs, history["val_loss"], label='Test loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('MNIST Loss'); plt.legend(); plt.ylim(0,1.05); plt.show()
plt.figure(); plt.plot(epochs, history["train_acc"], label='Train acc'); plt.plot(epochs, history["val_acc"], label='Test acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('MNIST Accuracy'); plt.legend(); plt.ylim(0,1); plt.show()

## Test set metrics

In [ ]:
def evaluate_classification_centralized(model, loader, device):
    model.eval().to(device)
    all_probs_list = [] # Use a list to append batches of probabilities
    all_labels_list = [] # Use a list to append batches of labels

    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            logits = model(Xb)
            # It's generally better to use torch.softmax
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs_list.append(probs)
            all_labels_list.append(yb.cpu().numpy()) # Ensure labels are also moved to CPU before converting to numpy

    all_probs  = np.concatenate(all_probs_list, axis=0)
    all_labels = np.concatenate(all_labels_list, axis=0)

    # Predicted classes
    preds = all_probs.argmax(axis=1)

    # Precision / Recall / F1
    print("Classification report:\n")
    print(classification_report(all_labels, preds, digits=4))

    # Confusion matrix
    print("Confusion matrix:\n")
    print(confusion_matrix(all_labels, preds))

    roc_auc = roc_auc_score(all_labels, all_probs,
                            multi_class='ovr', average='macro')

    pr_auc  = average_precision_score(all_labels, all_probs,
                                      average='macro')

    print(f"ROC-AUC (macro OvR):       {roc_auc:.4f}")
    print(f"PR-AUC (avg precision):    {pr_auc:.4f}")
    print()

print("=== Centralized Model Metrics ===")
evaluate_classification_centralized(model, test_loader, device)